In [13]:
from langgraph.graph import StateGraph,START,END
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from dotenv import load_dotenv
from typing import TypedDict
import os


In [14]:
load_dotenv()
token=os.getenv("Hugging_Face_Token")
print("Token loaded", token is not None)

Token loaded True


In [15]:
llm=HuggingFaceEndpoint(
     repo_id="Qwen/Qwen3-8B",
        task="text-generation",
        max_new_tokens=512,
        temperature=0.7,
        huggingfacehub_api_token=token,
)

In [16]:
model=ChatHuggingFace(llm=llm)

In [17]:
class Baseoutline(TypedDict):

    title:str
    outline:str
    blog:str

In [18]:
def CreateOutline(state:Baseoutline) ->Baseoutline:
    title=state['title']
    prompt=f"Create an outline based on the topic {title}"

    outline=model.invoke(prompt).content
    state['outline']=outline

    return state

In [19]:
def CreateBlog(state:Baseoutline) ->Baseoutline:
    title=state['title']
    outline=state['outline']
    prompt=f"Create an blog on the {title} based on the outline {outline}"

    blog=model.invoke(prompt).content
    state['blog']=blog

    return state
    

In [20]:
graph=StateGraph(Baseoutline)
graph.add_node('outline',CreateOutline)
graph.add_node('blog',CreateBlog)

graph.add_edge(START,'outline')
graph.add_edge('outline','blog')
graph.add_edge('blog',END)

workflow=graph.compile()

In [21]:
title={'title':'Mahatma Gandhi'}
answer=workflow.invoke(title)
print(answer['title'])
print(answer['outline'])
print(answer['blog'])

Mahatma Gandhi


**Outline: Mahatma Gandhi**  
**I. Introduction**  
- Brief overview of Mahatma Gandhi's significance as a global icon of peace, nonviolence, and civil rights.  
- His role in India’s independence movement and his lasting legacy.  
- Thesis: Gandhi’s philosophy of nonviolent resistance (

